# 36. Decode Scheduling | 解码调度
**难度：** Hard | **环境：** CPU-first | **标签：** `推理优化`, `解码`, `Scheduling` | **目标人群：** 推理优化学习者

---

## 本节导读

前面几节已经看过投机解码和多 Token 生成。现在把视角从单个请求扩展到一组同时生成的请求：有的刚完成输入处理，有的正在逐 Token 生成，有的即将结束。调度器需要根据这些状态决定下一步推进谁。

本节关注请求状态、选择依据和调度结果之间的关系：同一批请求采用不同的选择规则时，服务顺序、等待时间和生成进度会如何变化。

**关键词：** `decode scheduling`, `prefill`, `batch reordering`

---


## 前置阅读

**导语：** 进入本节前，先能区分请求的 prefill、decode 和完成状态，再观察调度器如何依据状态选择下一步请求。
- [21. Decoding Strategies | 解码策略](./21_Decoding_Strategies.ipynb)

---


### Step 1: 多请求为什么需要解码调度

把多个请求放进同一个等待队列后，它们会竞争下一轮 Decode 的执行机会。请求的阶段、优先级、缓存复用和已等待时间不同，服务顺序便会改变生成进度、响应延迟和公平性。

每一轮 Decode 都要在有限执行窗口中选择一部分 ready 请求。选择顺序会改变谁先推进、谁继续等待，以及整组请求的完成顺序；因此应同时观察等待时间、完成进度与公平性。

| 观察对象 | 要回答的问题 | 对调度的影响 |
| --- | --- | --- |
| 请求竞争 | 哪些请求正在等待下一轮推进？ | 形成可重排的活动集合 |
| 选择依据 | 为什么这次选择了某个请求？ | 由阶段、cache、优先级和长度共同决定 |
| 调度目标 | 希望改善哪类服务表现？ | 观察推进量、等待时间、完成顺序和公平性 |

![Decode 调度概念图：请求状态进入调度器并循环推进](../docs/public/02_PyTorch_Algorithms/36_decode_scheduling_board.svg)

### Step 2: 请求状态与生命周期机制

调度前先把一个请求看成一个会变化的状态对象。它从 prefill（处理输入）进入 decode（逐步生成），每次被选中后前进一小步，达到生成上限后离开活动集合；调度器的工作就是反复读取这些状态并推进其中一个请求。

| 机制环节 | 状态或动作 | 机制含义 |
|---|---|---|
| 状态对象 | 输入、阶段、生成进度、生成上限和缓存信息 | 说明请求当前处于什么阶段，还需要生成多少内容 |
| 状态转移 | prefill → decode | 请求完成输入处理，进入逐步生成阶段 |
| 活动集合 | 尚未完成的请求 | 只保留仍需要推进的请求，已完成请求退出队列 |
| 单步推进 | 选中一个请求前进一轮 | 每一轮只推进当前选中的请求，其他请求继续等待 |
| 完成条件 | 达到生成上限 | 请求完成并从活动集合中退出 |

### Step 3: 调度策略与排序优先级

有了请求生命周期，还需要一条选择规则来比较多个活动请求。调度策略把请求阶段、缓存状态、业务优先级和序列长度组织成有先后顺序的排序键；字段越靠前，决策优先级越高，排序键越小的请求越早获得推进机会。

下面的图展示排序字段的优先级，以及第一个不同字段如何决定两个请求的先后。排序规则把这些优先级固定为可重复比较的决策。

![Decode 调度排序键：从左到右比较字段](../docs/public/02_PyTorch_Algorithms/36_decode_sort_key.svg)

### Step 4: 实现请求状态与调度循环

请补全下方 `DecodeSchedulerSim`，跑通“入队 → 排序 → 单步推进 → 持续调度”这条链路。执行步数更新和输入检查已经给出，学习者只完成请求状态、排序键和本轮选择三个机制位置。

| TODO / 实现对象 | 学习者完成的机制 | 必须满足的约束 | 测试证据 |
|---|---|---|---|
| TODO 1 / `enqueue` | 构造请求状态并加入队列 | 默认从 `prefill` 开始，保留优先级和缓存命中状态 | 输入状态契约 |
| TODO 2 / `_schedule_key` | 将阶段、缓存、优先级和稳定字段组成排序键 | tuple 字段顺序必须与策略一致 | 排序与 tie-breaker |
| TODO 3 / `step` | 从 active 请求中选择并推进一个请求 | 完成请求退出；未选请求累计等待 | 状态转移、完成和等待 |
| `run` / 事件记录 | 使用已有循环推进并保存事件 | `max_steps` 只作循环保护 | 事件数量与异常输入 |



In [ ]:
from dataclasses import dataclass
from typing import Dict, List, Literal


In [ ]:
Phase = Literal['prefill', 'decode']


@dataclass
class RequestState:
    """记录一个请求在 prefill/decode 调度过程中的最小状态。"""
    request_id: int
    prompt_len: int
    max_new_tokens: int
    generated_len: int = 0
    phase: Phase = 'prefill'
    priority: int = 0
    cache_hit: bool = False
    wait_steps: int = 0  # 仅用于观察等待，不参与当前排序键

    @property
    def total_len(self) -> int:
        return self.prompt_len + self.generated_len

    @property
    def done(self) -> bool:
        return self.generated_len >= self.max_new_tokens


class DecodeSchedulerSim:
    """用事件序列观察多个请求如何在 prefill 与 decode 之间推进。"""

    def __init__(self):
        self.queue: List[RequestState] = []
        self.timeline: List[Dict[str, int | str]] = []

    def enqueue(self, request_id: int, prompt_len: int, max_new_tokens: int = 2, priority: int = 0, cache_hit: bool = False) -> None:
        """校验并登记请求；请求从 prefill 阶段开始。"""
        if request_id in {item.request_id for item in self.queue}:
            raise ValueError(f'重复的 request_id: {request_id}')
        if prompt_len <= 0 or max_new_tokens <= 0:
            raise ValueError('prompt_len 和 max_new_tokens 必须为正数')
        # ==========================================
        # TODO 1: 构造请求状态，并加入调度队列
        # 提示：RequestState 默认处于 prefill 阶段；把优先级和缓存命中状态一并登记。
        # request：待加入队列的 RequestState，保存请求的最小调度状态
        # ==========================================
        # request = ???
        self.queue.append(request)

    def _schedule_key(self, req: RequestState) -> tuple[int, int, int, int, int]:
        """返回当前示例策略的排序键；wait_steps 暂不参与排序。"""
        # ==========================================
        # TODO 2: 定义调度排序键
        # 提示：prefill 优先于 decode，cache hit 优先于 cache miss，高 priority 更靠前。
        # key：从左到右表达阶段、缓存命中、优先级和稳定 tie-breaker
        # ==========================================
        phase_rank = 0 if req.phase == 'prefill' else 1
        cache_rank = 0 if req.cache_hit else 1
        # key = ???
        return key

    def step(self) -> Dict[str, int | str] | None:
        """选择并推进一个活动请求，同时记录事件和等待步数。"""
        active = [req for req in self.queue if not req.done]
        if not active:
            return None

        # ==========================================
        # TODO 3: 从 active 请求中选择本轮要调度的请求
        # 提示：使用 min(..., key=self._schedule_key) 选择排序最靠前的请求。
        # chosen：本轮实际获得一个调度步的 RequestState
        # ==========================================
        # chosen = ???

        for req in active:
            if req is not chosen:
                req.wait_steps += 1

        if chosen.phase == 'prefill':
            chosen.phase = 'decode'
            action = 'prefill_to_decode'
        else:
            chosen.generated_len += 1
            action = 'finish' if chosen.done else 'decode_one_step'

        event = {
            'request_id': chosen.request_id,
            'phase': 'prefill' if action == 'prefill_to_decode' else 'decode',
            'action': action,
            'prompt_len': chosen.prompt_len,
            'generated_len': chosen.generated_len,
            'max_new_tokens': chosen.max_new_tokens,
            'wait_steps': chosen.wait_steps,
        }
        self.timeline.append(event)
        return event

    def run(self, max_steps: int = 100) -> List[Dict[str, int | str]]:
        """持续推进请求，直到全部完成或达到步数上限。"""
        if max_steps < 0:
            raise ValueError('max_steps 不能为负数')
        steps = 0
        while steps < max_steps:
            event = self.step()
            if event is None:
                break
            # 每个成功事件消耗一个调度步；这是 max_steps 的防护计数，不是调度策略。
            steps = steps + 1
        return self.timeline


In [ ]:
# 机制测试：分别验证请求状态、排序规则、推进过程和输入保护。
def test_enqueue_contract():
    """验证新请求从 prefill 状态入队，并保留输入契约字段。"""
    sim = DecodeSchedulerSim()
    sim.enqueue(request_id=1, prompt_len=2, max_new_tokens=2, priority=2, cache_hit=True)
    req = sim.queue[0]
    assert req.phase == 'prefill' and req.generated_len == 0
    assert req.prompt_len == 2 and req.max_new_tokens == 2

def test_schedule_key_order_and_tie_breaker():
    """验证阶段、缓存、优先级与 request_id 的排序关系。"""
    sim = DecodeSchedulerSim()
    sim.enqueue(9, 4, 1)
    sim.enqueue(3, 4, 1)
    assert sim._schedule_key(sim.queue[1]) < sim._schedule_key(sim.queue[0])
    assert sim.step()['request_id'] == 3

def test_state_progression_and_completion():
    """验证 prefill 到 decode、完成退出和等待累计。"""
    sim = DecodeSchedulerSim()
    sim.enqueue(1, 2, 2, priority=2, cache_hit=True)
    sim.enqueue(2, 3, 3, priority=1, cache_hit=False)
    events = sim.run(max_steps=20)
    assert len(events) == 7
    assert any(event['action'] == 'finish' for event in events)
    assert all(req.done for req in sim.queue)
    assert any(event['wait_steps'] > 0 for event in events)

def test_input_guards_and_step_limit():
    """验证非法请求与调度步数上限的保护。"""
    sim = DecodeSchedulerSim()
    for kwargs in ({'request_id': 1, 'prompt_len': 0}, {'request_id': 1, 'prompt_len': 1, 'max_new_tokens': 0}):
        try:
            sim.enqueue(**kwargs)
        except ValueError:
            continue
        raise AssertionError('非法请求参数应被拒绝')
    sim.enqueue(1, 1, 3)
    assert len(sim.run(max_steps=2)) == 2
    try:
        sim.run(max_steps=-1)
    except ValueError:
        pass
    else:
        raise AssertionError('负数 max_steps 应被拒绝')

def test_decode_scheduler():
    test_enqueue_contract()
    test_schedule_key_order_and_tie_breaker()
    test_state_progression_and_completion()
    test_input_guards_and_step_limit()
    print('✅ DecodeSchedulerSim 机制测试通过：状态、排序、推进和保护均通过。')


test_decode_scheduler()


## 参考代码与解析

### 代码

In [ ]:
# TODO：下面是题目区的参考实现。

Phase = Literal['prefill', 'decode']


@dataclass
class RequestState:
    """记录一个请求在 prefill/decode 调度过程中的最小状态。"""
    request_id: int
    prompt_len: int
    max_new_tokens: int
    generated_len: int = 0
    phase: Phase = 'prefill'
    priority: int = 0
    cache_hit: bool = False
    wait_steps: int = 0  # 仅用于观察等待，不参与当前排序键

    @property
    def total_len(self) -> int:
        return self.prompt_len + self.generated_len

    @property
    def done(self) -> bool:
        return self.generated_len >= self.max_new_tokens


class DecodeSchedulerSim:
    """用事件序列观察多个请求如何在 prefill 与 decode 之间推进。"""

    def __init__(self):
        self.queue: List[RequestState] = []
        self.timeline: List[Dict[str, int | str]] = []

    def enqueue(self, request_id: int, prompt_len: int, max_new_tokens: int = 2, priority: int = 0, cache_hit: bool = False) -> None:
        """校验并登记请求；请求从 prefill 阶段开始。"""
        if request_id in {item.request_id for item in self.queue}:
            raise ValueError(f'重复的 request_id: {request_id}')
        if prompt_len <= 0 or max_new_tokens <= 0:
            raise ValueError('prompt_len 和 max_new_tokens 必须为正数')
        # ==========================================
        # TODO 1: 构造请求状态，并加入调度队列
        # 提示: RequestState 默认处于 prefill 阶段；分别传入输入长度和生成上限
        # ==========================================
        request = RequestState(
            request_id=request_id,
            prompt_len=prompt_len,
            max_new_tokens=max_new_tokens,
            priority=priority,
            cache_hit=cache_hit,
        )
        self.queue.append(request)

    def _schedule_key(self, req: RequestState) -> tuple[int, int, int, int, int]:
        """返回当前示例策略的排序键；wait_steps 暂不参与排序。"""
        # ==========================================
        # TODO 2: 定义调度排序键
        # 提示: prefill 优先于 decode，cache hit 优先于 cache miss，高 priority 更靠前；
        #       tuple 会按从左到右的顺序比较字段
        # ==========================================
        phase_rank = 0 if req.phase == 'prefill' else 1
        cache_rank = 0 if req.cache_hit else 1
        key = (phase_rank, cache_rank, -req.priority, req.total_len, req.request_id)
        return key

    def step(self) -> Dict[str, int | str] | None:
        """选择并推进一个活动请求，同时记录事件和等待步数。"""
        active = [req for req in self.queue if not req.done]
        if not active:
            return None

        # ==========================================
        # TODO 3: 从 active 请求中选择本轮要调度的请求
        # 提示: 使用 min(..., key=self._schedule_key) 选择排序最靠前的请求
        # ==========================================
        chosen = min(active, key=self._schedule_key)
        for req in active:
            if req is not chosen:
                req.wait_steps += 1
        if chosen.phase == 'prefill':
            chosen.phase = 'decode'
            action = 'prefill_to_decode'
        else:
            chosen.generated_len += 1
            action = 'finish' if chosen.done else 'decode_one_step'

        event = {
            'request_id': chosen.request_id,
            'phase': 'prefill' if action == 'prefill_to_decode' else 'decode',
            'action': action,
            'prompt_len': chosen.prompt_len,
            'generated_len': chosen.generated_len,
            'max_new_tokens': chosen.max_new_tokens,
            'wait_steps': chosen.wait_steps,
        }
        self.timeline.append(event)
        return event

    def run(self, max_steps: int = 100) -> List[Dict[str, int | str]]:
        """持续推进请求，直到全部完成或达到步数上限。"""
        if max_steps < 0:
            raise ValueError('max_steps 不能为负数')
        steps = 0
        while steps < max_steps:
            event = self.step()
            if event is None:
                break
            # 每个成功事件消耗一个调度步；这是 max_steps 的防护计数，不是调度策略。
            steps = steps + 1
        return self.timeline


### 解析

本题依次实现请求状态登记、调度排序键和单步推进。答案代码保留与题目区相同的控制流，只补全每处机制决策。

**TODO 1：构造请求状态**

- 使用 `RequestState` 保存输入长度、生成上限、阶段、优先级和 cache 命中状态。
- 新请求默认处于 `prefill`，`generated_len` 从 0 开始，再加入调度队列。

**TODO 2：定义调度排序键**

- 使用 `key = (phase_rank, cache_rank, -req.priority, req.total_len, req.request_id)` 组织排序字段。
- 排序键越小越优先：`prefill` 用 0、`decode` 用 1，cache hit 用 0、cache miss 用 1；负的 priority 让高优先级请求靠前。
- `total_len` 和 `request_id` 只负责稳定打破平局，不改变前面的调度优先级。

**TODO 3：选择本轮调度请求**

- 使用 `chosen = min(active, key=self._schedule_key)`，每一步都根据当前状态重新选择请求。
- `active` 已过滤完成请求；未被选中的活动请求增加 `wait_steps`，用于观察等待情况。
- 选中 `prefill` 请求时切换到 `decode`；选中 `decode` 请求时增加 `generated_len`，达到上限后记录完成事件。

**测试如何定位问题**

- 输入契约测试检查非法长度和重复 `request_id`；排序测试检查优先级与稳定 tie-breaker。
- 推进测试检查阶段转换、完成事件和等待步数，帮助定位状态更新或选择规则的问题。

### Step 5: 可选 GPU 微基准

#### 5.1 环境、请求与固定条件

本步用人工构造的 GPU 矩阵乘法负载比较 FIFO 与本节排序键，观察完成顺序、等待步数、执行时间和峰值显存。运行前确认当前 Notebook 使用目标 GPU 环境；也可在仓库根目录执行 `python tools/environment_preflight.py --gpu --packages torch`。

配置单元集中保存 dtype、请求规格、重复次数和结果路径；执行单元自动创建 JSON。真实服务的 TTFT、TPOT 与 backend 对照在 [70. Serving Scheduler Benchmark](./70_Serving_Scheduler_Benchmark.ipynb) 中记录。

#### 5.2 配置与执行

| 实验内容 | 固定或改变的条件 | 观察目的 |
|---|---|---|
| GPU 负载 | 人工构造的矩阵乘法；`GPU_HIDDEN_SIZE` 控制规模 | 观察调度执行的 GPU 时间和峰值显存 |
| 请求集合 | `REQUEST_SPECS` 固定输入长度、生成上限、优先级和缓存状态 | 保持两种策略使用同一批请求 |
| 对照变量 | FIFO vs 本节排序键 | 比较请求完成顺序和等待步数 |
| 运行条件 | dtype、重复次数、随机种子保持一致 | 让结果可以复查 |
| 结果指标 | 总耗时、token/s、完成步、`wait_steps`、峰值显存、OOM | 判断人工负载下的调度代价；evidence level 为 `synthetic_gpu_microbenchmark` |



In [ ]:
# Step 5 配置单元：先运行本 cell，再运行下面的 GPU 执行单元。
RUN_GPU_MICROBENCH = False  # False 只检查代码；True 才运行 GPU 并保存结果。
GPU_RESULT_PATH = 'benchmarks/results/36_decode_scheduling_gpu.json'  # 相对项目根目录，无需填写绝对路径。
GPU_REPEATS = 3  # 两种策略使用相同重复次数。
GPU_HIDDEN_SIZE = 512  # 人工构造矩阵的宽度；只改变 workload 时再修改。
GPU_DTYPE = 'float16'  # 可改为 bfloat16，但必须确认硬件原生支持。
GPU_SEED = 42
REQUEST_SPECS = [
    {'request_id': 1, 'prompt_len': 64, 'max_new_tokens': 16, 'priority': 1, 'cache_hit': False},
    {'request_id': 2, 'prompt_len': 128, 'max_new_tokens': 16, 'priority': 2, 'cache_hit': True},
    {'request_id': 3, 'prompt_len': 256, 'max_new_tokens': 16, 'priority': 1, 'cache_hit': False},
]


In [ ]:
# Step 5 GPU 执行单元：依赖上面的配置单元；默认不启动 GPU。
def _run_gpu_schedule_policy(policy):
    import time
    import torch

    if not torch.cuda.is_available():
        raise RuntimeError('GPU 微基准需要 CUDA；请先切换到 GPU runtime。')
    if GPU_DTYPE not in {'float16', 'bfloat16'}:
        raise ValueError('GPU_DTYPE 只能是 float16 或 bfloat16')
    if GPU_DTYPE == 'bfloat16' and not torch.cuda.is_bf16_supported(including_emulation=False):
        raise RuntimeError('当前 GPU 不支持原生 BF16，请改用 float16。')
    torch.manual_seed(GPU_SEED)
    torch.cuda.manual_seed_all(GPU_SEED)
    device = torch.device('cuda')
    dtype = getattr(torch, GPU_DTYPE)
    weight = torch.randn(GPU_HIDDEN_SIZE, GPU_HIDDEN_SIZE, device=device, dtype=dtype)
    inputs = {spec['request_id']: torch.randn(1, GPU_HIDDEN_SIZE, device=device, dtype=dtype) for spec in REQUEST_SPECS}
    generated = {spec['request_id']: 0 for spec in REQUEST_SPECS}
    wait_steps = {spec['request_id']: 0 for spec in REQUEST_SPECS}
    completion_step = {}
    total_events = sum(spec['max_new_tokens'] + 1 for spec in REQUEST_SPECS)

    for _ in range(2):
        _ = inputs[REQUEST_SPECS[0]['request_id']] @ weight
    torch.cuda.synchronize()
    torch.cuda.reset_peak_memory_stats()
    sim = DecodeSchedulerSim() if policy == 'priority_cache_aware' else None
    if sim is not None:
        for spec in REQUEST_SPECS:
            sim.enqueue(**spec)
    torch.cuda.synchronize()
    start = time.perf_counter()
    for step_index in range(total_events):
        if policy == 'fifo':
            request_id = next(
                spec['request_id'] for spec in REQUEST_SPECS
                if spec['request_id'] not in completion_step
            )
            spec = next(item for item in REQUEST_SPECS if item['request_id'] == request_id)
            if generated[request_id] < spec['max_new_tokens']:
                generated[request_id] += 1
            else:
                completion_step[request_id] = step_index + 1
        else:
            event = sim.step()
            if event is None:
                break
            request_id = event['request_id']
            generated[request_id] = event['generated_len']
            wait_steps[request_id] = event['wait_steps']
            if event['action'] == 'finish':
                completion_step[request_id] = step_index + 1
        _ = inputs[request_id] @ weight
        if policy == 'fifo':
            for other_id in wait_steps:
                if other_id != request_id and other_id not in completion_step:
                    wait_steps[other_id] += 1
    torch.cuda.synchronize()
    elapsed = time.perf_counter() - start
    total_tokens = sum(generated.values())
    return {
        'policy': policy,
        'dtype': GPU_DTYPE,
        'elapsed_ms': round(elapsed * 1000, 3),
        'tokens_per_s': round(total_tokens / elapsed, 3) if elapsed else 0.0,
        'completion_step': completion_step,
        'wait_steps': wait_steps,
        'peak_memory_mb': round(torch.cuda.max_memory_allocated() / (1024 ** 2), 2),
    }

if RUN_GPU_MICROBENCH:
    import json
    from pathlib import Path

    gpu_results = {
        policy: [_run_gpu_schedule_policy(policy) for _ in range(GPU_REPEATS)]
        for policy in ('fifo', 'priority_cache_aware')
    }
    # 自动向上查找包含 benchmarks/ 的项目根目录，兼容从仓库根目录或 Notebook 子目录启动。
    project_root = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'benchmarks').is_dir()), Path.cwd())
    output_path = project_root / GPU_RESULT_PATH
    output_path.parent.mkdir(parents=True, exist_ok=True)
    report = {
        'task': 'decode_scheduling_gpu_microbenchmark',
        'evidence_level': 'synthetic_gpu_microbenchmark',
        'config': {
            'workload': 'synthetic_matmul',
            'request_count': len(REQUEST_SPECS),
            'request_specs': REQUEST_SPECS,
            'hidden_size': GPU_HIDDEN_SIZE,
            'dtype': GPU_DTYPE,
            'repeats': GPU_REPEATS,
            'seed': GPU_SEED,
            'torch': torch.__version__,
            'torch_cuda': torch.version.cuda,
            'device': torch.cuda.get_device_name(0),
        },
        'results': gpu_results,
        'decision': {'decision': 'measure', 'reason': '仅比较人工构造的 GPU 矩阵乘法负载下的调度顺序和执行开销。'},
    }
    output_path.write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding='utf-8')
    print(f'GPU 微基准结果已保存：{output_path}')
    print(json.dumps(report, ensure_ascii=False, indent=2))
else:
    print('GPU 微基准未启动：将 RUN_GPU_MICROBENCH 改为 True 后运行。')


#### 5.3 读取结果与记录证据

读取执行单元生成的 JSON，先核对 workload 和 evidence level，再填写下方记录表。没有实际运行结果时保留待复测状态。

In [ ]:
# 5.3：读取 GPU 微基准 JSON；默认没有结果时保留待复测状态。
import json
from pathlib import Path
result_path = Path(GPU_RESULT_PATH)
if result_path.exists():
    report = json.loads(result_path.read_text(encoding='utf-8'))
    required = {'task', 'evidence_level', 'config', 'results', 'decision'}
    missing = required - set(report)
    if missing:
        raise ValueError(f'结果 JSON 缺少字段：{sorted(missing)}')
    print(report)
else:
    print(f'尚无 GPU 微基准结果：{result_path}')


#### GPU 实验记录

完成 GPU 执行单元后填写下面这张总表。共同条件写入第三列，FIFO 与本节排序键分别填写对应结果；调整请求规格后新增一组记录，不覆盖已有结果。

| 类别 | 项目 | 共同条件 / 本次配置 | FIFO | 本节排序键 | 说明 |
|---|---|---|---:|---:|---|
| 环境 | GPU 工作负载 | 人工构造的 GPU 矩阵乘法；`GPU_HIDDEN_SIZE` 控制矩阵规模 |  |  | 记录 GPU、PyTorch / CUDA 和矩阵规模 |
| 环境 | dtype | `GPU_DTYPE`；默认 `float16` |  |  | BF16 需确认硬件原生支持 |
| workload | 请求规格 | `REQUEST_SPECS` 固定请求数、输入长度、优先级和 `max_new_tokens` |  |  | 改变请求规格时新增一组记录 |
| workload | 重复与随机性 | `GPU_REPEATS`、`GPU_SEED` |  |  | 两种策略使用相同设置 |
| 结果 | 总耗时（ms） | 预热后重复运行 |  |  | 越低越好，但需结合等待和完成步数 |
| 结果 | 吞吐（token/s） | 固定输出 token 数 |  |  | 仅表示本人工构造负载的执行吞吐 |
| 结果 | 最晚完成步 / 最大 `wait_steps` | 从事件记录计算 |  |  | 观察服务顺序和等待情况 |
| 结果 | 峰值显存（MB） / OOM | 从 CUDA 统计和运行状态记录 |  |  | OOM 时保留失败状态 |
| 结果 | JSON 报告 | `GPU_RESULT_PATH` |  |  | 每组实验使用独立路径 |


## 相关阅读

完成请求状态、选择规则和推进顺序的模拟后，可以继续阅读推理服务调度的论文与真实基准项目。

- [Orca 原论文：A Distributed Serving System for Transformer-Based Generative Models](https://www.usenix.org/conference/osdi22/presentation/yu)
- [vLLM 官方仓库](https://github.com/vllm-project/vllm)
- [37. KV Cache Scheduling | KV Cache 调度](./37_KV_Cache_Scheduling.ipynb)
- [38. Prefill/Decode Scheduling | Prefill/Decode 调度](./38_Prefill_Decode_Scheduling.ipynb)
- [70. Serving Scheduler Benchmark | 服务调度基准项目](./70_Serving_Scheduler_Benchmark.ipynb)
